# Plot Regression MC v2

# 1. Import Library

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.ticker import FuncFormatter

from scipy.ndimage import gaussian_filter

plt.rcParams.update({
    'font.family': 'Helvetica',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
})

CACHE_DIR = Path('../../data/intermediate/divided_regression_1991_2020')
RESULTS_DIR = Path('../../results/analisis_regresi_26-19_v2')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

mc_extent = [80, 160, -20, 20]
mc_xticks = np.arange(90, 161, 20)
mc_yticks = np.arange(-20, 21, 10)

wind_step = 8
mfc_step = 10
psi_chi_step = 8

DIVERGING_CMAP = 'bwr'


def save_plot(fig, filename):
    fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches='tight')


def subset_for_extent(da, extent=mc_extent):
    """Subset DataArray to map domain before estimating plot scales."""
    if not isinstance(da, xr.DataArray) or not {'lat', 'lon'}.issubset(da.dims):
        return da

    lon_min, lon_max, lat_min, lat_max = extent
    out = da

    lon_vals = out['lon'].values
    lat_vals = out['lat'].values

    if lon_vals[0] <= lon_vals[-1]:
        out = out.sel(lon=slice(lon_min, lon_max))
    else:
        out = out.sel(lon=slice(lon_max, lon_min))

    if lat_vals[0] <= lat_vals[-1]:
        out = out.sel(lat=slice(lat_min, lat_max))
    else:
        out = out.sel(lat=slice(lat_max, lat_min))

    return out


def smooth_for_plot(da, sigma=0.7):
    """NaN-safe Gaussian smoothing for display only."""
    if sigma is None or sigma <= 0 or not {'lat', 'lon'}.issubset(da.dims):
        return da

    def _smooth(arr):
        mask = np.isfinite(arr)
        filled = np.where(mask, arr, 0.0)
        weight = gaussian_filter(mask.astype(float), sigma=sigma)
        smoothed = gaussian_filter(filled, sigma=sigma)
        with np.errstate(invalid='ignore', divide='ignore'):
            out = smoothed / weight
        out[weight <= 0] = np.nan
        return out

    smoothed = xr.apply_ufunc(
        _smooth,
        da,
        input_core_dims=[['lat', 'lon']],
        output_core_dims=[['lat', 'lon']],
        vectorize=True,
        dask='allowed',
        output_dtypes=[da.dtype],
    )
    smoothed = smoothed.transpose(*da.dims)
    smoothed = smoothed.assign_attrs(da.attrs)
    if da.name is not None:
        smoothed = smoothed.rename(da.name)
    return smoothed


def _decimal_places(step):
    """Decimal places needed to print tick labels cleanly."""
    if not np.isfinite(step) or step <= 0:
        return 0
    if step >= 1:
        return 0 if np.isclose(step, round(step)) else 1
    return int(np.ceil(-np.log10(step))) + 1


def format_nice_number(value, pos=None):
    """Compact labels for rounded ticks; small values use readable scientific notation."""
    if not np.isfinite(value):
        return ''
    if np.isclose(value, 0.0):
        return '0'

    abs_value = abs(value)
    if abs_value < 1e-2 or abs_value >= 1e4:
        exponent = int(np.floor(np.log10(abs_value)))
        mantissa = value / (10.0 ** exponent)
        if np.isclose(mantissa, round(mantissa), rtol=1e-6, atol=1e-12):
            mantissa_text = f'{mantissa:.0f}'
        else:
            mantissa_text = f'{mantissa:.1f}'.rstrip('0').rstrip('.')
        return f'{mantissa_text}e{exponent}'

    if abs_value >= 100:
        text = f'{value:.0f}'
    elif abs_value >= 10:
        text = f'{value:.1f}'
    elif abs_value >= 1:
        text = f'{value:.2f}'
    else:
        text = f'{value:.3f}'

    return text.rstrip('0').rstrip('.')


def style_horizontal_cbar(cbar):
    cbar.ax.xaxis.set_major_formatter(FuncFormatter(format_nice_number))
    cbar.update_ticks()
    return cbar


def nice_step(value, mode='ceil', bases=(1.0, 2.0, 2.5, 5.0, 10.0)):
    """
    Rounded interval selector.
    Bases exclude odd values like 1.25, 1.5, 3, 4, and 7.5 so colorbar ticks stay neat.
    """
    if not np.isfinite(value) or value <= 0:
        return 1.0

    exponent = np.floor(np.log10(value))
    fraction = value / (10.0 ** exponent)
    allowed = np.asarray(bases, dtype=float)

    if mode == 'ceil':
        idx = np.searchsorted(allowed, fraction, side='left')
        if idx >= len(allowed):
            idx = len(allowed) - 1
    elif mode == 'floor':
        idx = np.searchsorted(allowed, fraction, side='right') - 1
        idx = max(idx, 0)
    else:
        idx = int(np.argmin(np.abs(np.log(allowed / fraction))))

    return float(allowed[idx] * (10.0 ** exponent))


def _round_values(values, step):
    decimals = _decimal_places(step)
    return np.round(values, decimals + 2)


def _finite_values(*arrays, extent=mc_extent):
    values = []
    for arr in arrays:
        if arr is None:
            continue
        arr = subset_for_extent(arr, extent=extent)
        vals = np.asarray(arr).ravel()
        vals = vals[np.isfinite(vals)]
        if vals.size:
            values.append(vals)
    if not values:
        return np.array([1.0])
    return np.concatenate(values)


def symmetric_levels(
    *arrays,
    n_levels=29,
    n_ticks=13,
    quantile=0.975,
    padding=1.05,
    extent=mc_extent,
):
    """
    Symmetric diverging levels using a robust percentile, then rounded tick steps.
    This keeps colors from becoming pale without creating awkward tick labels.
    """
    values = _finite_values(*arrays, extent=extent)
    abs_values = np.abs(values[np.isfinite(values)])
    ref = float(np.nanquantile(abs_values, quantile)) if abs_values.size else 1.0

    if not np.isfinite(ref) or np.isclose(ref, 0.0):
        ref = float(np.nanmax(abs_values)) if abs_values.size else 1.0
    if not np.isfinite(ref) or np.isclose(ref, 0.0):
        ref = 1.0

    raw_bound = ref * padding

    # Tick interval is rounded and used to set the plotted bound.
    tick_step = nice_step((2.0 * raw_bound) / max(n_ticks - 1, 1), mode='ceil')
    bound = tick_step * np.ceil(raw_bound / tick_step)

    # Contour/color levels are denser than major ticks but still based on neat intervals.
    raw_level_step = (2.0 * bound) / max(n_levels - 1, 1)
    level_step = nice_step(raw_level_step, mode='round')
    if level_step >= tick_step:
        level_step = tick_step / 2.0

    levels = np.arange(-bound, bound + level_step * 0.5, level_step)
    ticks = np.arange(-bound, bound + tick_step * 0.5, tick_step)

    levels = _round_values(levels, level_step)
    ticks = _round_values(ticks, tick_step)
    return levels, ticks, bound


def split_period_levels(all_data, p1_data, p2_data, diff_data, **kwargs):
    """Use one scale for all/P1/P2 and a smaller independent scale for P2-P1."""
    main_kwargs = dict(kwargs)
    diff_kwargs = dict(kwargs)
    diff_kwargs['quantile'] = kwargs.get('diff_quantile', kwargs.get('quantile', 0.985))
    main_kwargs.pop('diff_quantile', None)
    diff_kwargs.pop('diff_quantile', None)

    main_levels, main_ticks, main_bound = symmetric_levels(all_data, p1_data, p2_data, **main_kwargs)
    diff_levels, diff_ticks, diff_bound = symmetric_levels(diff_data, **diff_kwargs)
    return {
        'main': {'levels': main_levels, 'ticks': main_ticks, 'bound': main_bound},
        'diff': {'levels': diff_levels, 'ticks': diff_ticks, 'bound': diff_bound},
    }


def vector_style(
    cases,
    unit_label,
    quantile=0.90,
    key_fraction=0.50,
    target_ref_width=0.085,
    extent=mc_extent,
):
    """
    Robust quiver scale.

    ref is the representative vector magnitude.
    target_ref_width controls data-arrow length.
    key_fraction controls the quiver-key magnitude, so the key stays shorter than the field arrows.
    """
    magnitudes = []
    for u_data, v_data in cases:
        u = subset_for_extent(u_data, extent=extent)
        v = subset_for_extent(v_data, extent=extent)
        vals = np.hypot(np.asarray(u), np.asarray(v)).ravel()
        vals = vals[np.isfinite(vals)]
        if vals.size:
            magnitudes.append(vals)

    if magnitudes:
        merged = np.concatenate(magnitudes)
        ref = float(np.nanquantile(merged, quantile))
    else:
        ref = 1.0
    if not np.isfinite(ref) or np.isclose(ref, 0.0):
        ref = 1.0

    key_u = nice_step(ref * key_fraction, mode='round')
    key_u = max(key_u, 1e-30)

    # With scale_units='width', arrow length is roughly U / scale in axis-width units.
    scale = ref / target_ref_width
    key_label = f'{format_nice_number(key_u)} {unit_label}'
    return scale, key_u, key_label


def split_vector_styles(main_cases, diff_cases, unit_label, **kwargs):
    return {
        'main': vector_style(main_cases, unit_label, **kwargs),
        'diff': vector_style(diff_cases, unit_label, **kwargs),
    }


def draw_base_map(ax):
    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_xlabel('')
    ax.set_ylabel('')


def add_colorbar(fig, ax, img, ticks, label):
    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend='both')
    style_horizontal_cbar(cbar)
    cbar.set_label(label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=13)
    return cbar


def plot_vector_period_maps(
    plot_defs,
    level_sets,
    vector_styles,
    cbar_label,
    smooth_sigma=0.7,
    quiver_step=wind_step,
    cmap=DIVERGING_CMAP,
):
    for shade_data, u_data, v_data, period_key, title, filename in plot_defs:
        fig = plt.figure(figsize=(10, 6))
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

        plot_data = smooth_for_plot(shade_data, sigma=smooth_sigma).reset_coords(drop=True)
        img = plot_data.plot.pcolormesh(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            levels=level_sets[period_key]['levels'],
            extend='both',
            add_colorbar=False,
            add_labels=False,
            infer_intervals=True,
        )

        u_domain = subset_for_extent(u_data)
        v_domain = subset_for_extent(v_data)
        u_plot = u_domain.isel(lat=slice(None, None, quiver_step), lon=slice(None, None, quiver_step))
        v_plot = v_domain.isel(lat=slice(None, None, quiver_step), lon=slice(None, None, quiver_step))
        scale, key_u, key_label = vector_styles[period_key]

        q = ax.quiver(
            u_plot['lon'].values,
            u_plot['lat'].values,
            u_plot.values,
            v_plot.values,
            transform=ccrs.PlateCarree(),
            scale=scale,
            scale_units='width',
            width=0.0020,
            headwidth=3.5,
            headlength=4.5,
            headaxislength=4.0,
            color='black',
            zorder=4,
        )
        ax.quiverkey(
            q,
            X=0.84,
            Y=1.055,
            U=key_u,
            label=key_label,
            labelpos='E',
            coordinates='axes',
            color='black',
            fontproperties={'size': 10.5},
        )

        draw_base_map(ax)
        ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
        add_colorbar(fig, ax, img, level_sets[period_key]['ticks'], cbar_label)

        save_plot(fig, filename)
        plt.show()
        plt.close(fig)


def plot_scalar_period_maps(
    plot_defs,
    level_sets,
    cbar_label,
    smooth_sigma=0.5,
    sig_step=8,
    cmap=DIVERGING_CMAP,
):
    for data, sig_mask, period_key, title, filename in plot_defs:
        fig = plt.figure(figsize=(10, 6))
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

        plot_data = smooth_for_plot(data, sigma=smooth_sigma).reset_coords(drop=True)
        img = plot_data.plot.pcolormesh(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            levels=level_sets[period_key]['levels'],
            extend='both',
            add_colorbar=False,
            add_labels=False,
            infer_intervals=True,
        )

        if sig_mask is not None:
            sig_domain = subset_for_extent(sig_mask.fillna(False).astype(np.int8))
            sig_plot = sig_domain.coarsen(lat=sig_step, lon=sig_step, boundary='trim').max() > 0
            yy, xx = np.where(sig_plot.values)
            if yy.size > 0:
                ax.scatter(
                    sig_plot['lon'].values[xx],
                    sig_plot['lat'].values[yy],
                    s=12,
                    c='k',
                    marker='.',
                    linewidths=0,
                    alpha=0.65,
                    transform=ccrs.PlateCarree(),
                    zorder=4,
                )

        draw_base_map(ax)
        ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
        add_colorbar(fig, ax, img, level_sets[period_key]['ticks'], cbar_label)

        save_plot(fig, filename)
        plt.show()
        plt.close(fig)


# 2. Load Cache

In [ ]:
# rain_ds = xr.open_dataset(CACHE_DIR / 'rainfall_reg_cache_1981_2025.nc')
wind_ds = xr.open_dataset(CACHE_DIR / 'wind_reg_cache_1981_2025.nc')
mfc_ds = xr.open_dataset(CACHE_DIR / 'mfc_reg_cache_1981_2025.nc')
psi_ds = xr.open_dataset(CACHE_DIR / 'psi_reg_cache_1981_2025.nc')
chi_ds = xr.open_dataset(CACHE_DIR / 'chi_reg_cache_1981_2025.nc')
mslp_ds = xr.open_dataset(CACHE_DIR / 'mslp_reg_cache_1981_2025.nc')
gh850_ds = xr.open_dataset(CACHE_DIR / 'gh850_reg_cache_1981_2025.nc')

# Rainfall is skipped in v2 until the new build is ready.
# rain_reg_clim = rain_ds['rain_reg_clim']
# rain_reg_past = rain_ds['rain_reg_past']
# rain_reg_recent = rain_ds['rain_reg_recent']
# rain_reg_recent_minus_past = rain_ds['rain_reg_recent_minus_past']
# rain_reg_clim_sig = rain_ds['rain_reg_clim_sig']
# rain_reg_past_sig = rain_ds['rain_reg_past_sig']
# rain_reg_recent_sig = rain_ds['rain_reg_recent_sig']
# rain_reg_recent_minus_past_sig = rain_ds['rain_reg_recent_minus_past_sig']

wind_u_reg_clim = wind_ds['wind_u_reg_clim']
wind_u_reg_past = wind_ds['wind_u_reg_past']
wind_u_reg_recent = wind_ds['wind_u_reg_recent']
wind_u_reg_recent_minus_past = wind_ds['wind_u_reg_recent_minus_past']
wind_u_reg_clim_sig = wind_ds['wind_u_reg_clim_sig']
wind_u_reg_past_sig = wind_ds['wind_u_reg_past_sig']
wind_u_reg_recent_sig = wind_ds['wind_u_reg_recent_sig']

wind_v_reg_clim = wind_ds['wind_v_reg_clim']
wind_v_reg_past = wind_ds['wind_v_reg_past']
wind_v_reg_recent = wind_ds['wind_v_reg_recent']
wind_v_reg_recent_minus_past = wind_ds['wind_v_reg_recent_minus_past']
wind_v_reg_clim_sig = wind_ds['wind_v_reg_clim_sig']
wind_v_reg_past_sig = wind_ds['wind_v_reg_past_sig']
wind_v_reg_recent_sig = wind_ds['wind_v_reg_recent_sig']
wind_vector_sig_clim = wind_ds['wind_vector_sig_clim']
wind_vector_sig_past = wind_ds['wind_vector_sig_past']
wind_vector_sig_recent = wind_ds['wind_vector_sig_recent']
wind_vector_sig_recent_minus_past = wind_ds['wind_vector_sig_recent_minus_past']
wind_u_reg_recent_minus_past_sig = wind_ds['wind_u_reg_recent_minus_past_sig']
wind_v_reg_recent_minus_past_sig = wind_ds['wind_v_reg_recent_minus_past_sig']

mfc_reg_clim = mfc_ds['mfc_reg_clim']
mfc_reg_past = mfc_ds['mfc_reg_past']
mfc_reg_recent = mfc_ds['mfc_reg_recent']
mfc_reg_recent_minus_past = mfc_ds['mfc_reg_recent_minus_past']
mfc_reg_clim_sig = mfc_ds['mfc_reg_clim_sig']
mfc_reg_past_sig = mfc_ds['mfc_reg_past_sig']
mfc_reg_recent_sig = mfc_ds['mfc_reg_recent_sig']
mfc_reg_recent_minus_past_sig = mfc_ds['mfc_reg_recent_minus_past_sig']
mfc_qx_reg_clim = mfc_ds['mfc_qx_reg_clim']
mfc_qx_reg_past = mfc_ds['mfc_qx_reg_past']
mfc_qx_reg_recent = mfc_ds['mfc_qx_reg_recent']
mfc_qx_reg_recent_minus_past = mfc_ds['mfc_qx_reg_recent_minus_past']
mfc_qx_reg_clim_sig = mfc_ds['mfc_qx_reg_clim_sig']
mfc_qx_reg_past_sig = mfc_ds['mfc_qx_reg_past_sig']
mfc_qx_reg_recent_sig = mfc_ds['mfc_qx_reg_recent_sig']
mfc_qy_reg_clim = mfc_ds['mfc_qy_reg_clim']
mfc_qy_reg_past = mfc_ds['mfc_qy_reg_past']
mfc_qy_reg_recent = mfc_ds['mfc_qy_reg_recent']
mfc_qy_reg_recent_minus_past = mfc_ds['mfc_qy_reg_recent_minus_past']
mfc_qy_reg_clim_sig = mfc_ds['mfc_qy_reg_clim_sig']
mfc_qy_reg_past_sig = mfc_ds['mfc_qy_reg_past_sig']
mfc_qy_reg_recent_sig = mfc_ds['mfc_qy_reg_recent_sig']
mfc_vector_sig_clim = mfc_ds['mfc_vector_sig_clim']
mfc_vector_sig_past = mfc_ds['mfc_vector_sig_past']
mfc_vector_sig_recent = mfc_ds['mfc_vector_sig_recent']
mfc_vector_sig_recent_minus_past = mfc_ds['mfc_vector_sig_recent_minus_past']

psi_reg_clim = psi_ds['psi_reg_clim']
psi_reg_past = psi_ds['psi_reg_past']
psi_reg_recent = psi_ds['psi_reg_recent']
psi_reg_recent_minus_past = psi_ds['psi_reg_recent_minus_past']
psi_reg_clim_sig = psi_ds['psi_reg_clim_sig']
psi_reg_past_sig = psi_ds['psi_reg_past_sig']
psi_reg_recent_sig = psi_ds['psi_reg_recent_sig']
psi_reg_recent_minus_past_sig = psi_ds['psi_reg_recent_minus_past_sig']
u_psi_reg_clim = psi_ds['u_psi_reg_clim']
u_psi_reg_past = psi_ds['u_psi_reg_past']
u_psi_reg_recent = psi_ds['u_psi_reg_recent']
u_psi_reg_recent_minus_past = psi_ds['u_psi_reg_recent_minus_past']
v_psi_reg_clim = psi_ds['v_psi_reg_clim']
v_psi_reg_past = psi_ds['v_psi_reg_past']
v_psi_reg_recent = psi_ds['v_psi_reg_recent']
v_psi_reg_recent_minus_past = psi_ds['v_psi_reg_recent_minus_past']
psi_vector_sig_clim = psi_ds['psi_vector_sig_clim']
psi_vector_sig_past = psi_ds['psi_vector_sig_past']
psi_vector_sig_recent = psi_ds['psi_vector_sig_recent']
psi_vector_sig_recent_minus_past = psi_ds['psi_vector_sig_recent_minus_past']

chi_reg_clim = chi_ds['chi_reg_clim']
chi_reg_past = chi_ds['chi_reg_past']
chi_reg_recent = chi_ds['chi_reg_recent']
chi_reg_recent_minus_past = chi_ds['chi_reg_recent_minus_past']
chi_reg_clim_sig = chi_ds['chi_reg_clim_sig']
chi_reg_past_sig = chi_ds['chi_reg_past_sig']
chi_reg_recent_sig = chi_ds['chi_reg_recent_sig']
chi_reg_recent_minus_past_sig = chi_ds['chi_reg_recent_minus_past_sig']
u_chi_reg_clim = chi_ds['u_chi_reg_clim']
u_chi_reg_past = chi_ds['u_chi_reg_past']
u_chi_reg_recent = chi_ds['u_chi_reg_recent']
u_chi_reg_recent_minus_past = chi_ds['u_chi_reg_recent_minus_past']
v_chi_reg_clim = chi_ds['v_chi_reg_clim']
v_chi_reg_past = chi_ds['v_chi_reg_past']
v_chi_reg_recent = chi_ds['v_chi_reg_recent']
v_chi_reg_recent_minus_past = chi_ds['v_chi_reg_recent_minus_past']
chi_vector_sig_clim = chi_ds['chi_vector_sig_clim']
chi_vector_sig_past = chi_ds['chi_vector_sig_past']
chi_vector_sig_recent = chi_ds['chi_vector_sig_recent']
chi_vector_sig_recent_minus_past = chi_ds['chi_vector_sig_recent_minus_past']

mslp_reg_clim = mslp_ds['mslp_reg_clim']
mslp_reg_past = mslp_ds['mslp_reg_past']
mslp_reg_recent = mslp_ds['mslp_reg_recent']
mslp_reg_recent_minus_past = mslp_ds['mslp_reg_recent_minus_past']
mslp_reg_clim_sig = mslp_ds['mslp_reg_clim_sig']
mslp_reg_past_sig = mslp_ds['mslp_reg_past_sig']
mslp_reg_recent_sig = mslp_ds['mslp_reg_recent_sig']
mslp_reg_recent_minus_past_sig = mslp_ds['mslp_reg_recent_minus_past_sig']

gh850_reg_clim = gh850_ds['gh850_reg_clim']
gh850_reg_past = gh850_ds['gh850_reg_past']
gh850_reg_recent = gh850_ds['gh850_reg_recent']
gh850_reg_recent_minus_past = gh850_ds['gh850_reg_recent_minus_past']
gh850_reg_clim_sig = gh850_ds['gh850_reg_clim_sig']
gh850_reg_past_sig = gh850_ds['gh850_reg_past_sig']
gh850_reg_recent_sig = gh850_ds['gh850_reg_recent_sig']
gh850_reg_recent_minus_past_sig = gh850_ds['gh850_reg_recent_minus_past_sig']

# 3. Rainfall Regression (skipped in v2)

In [ ]:
# plot_defs = [
#     (rain_reg_clim, rain_reg_clim_sig, 'Regresi Rainfall vs Nino34 DJF 1981-2025', 'rainfall_reg_1981-2025.png'),
#     (rain_reg_past, rain_reg_past_sig, 'Regresi Rainfall vs Nino34 DJF P1', 'rainfall_reg_p1.png'),
#     (rain_reg_recent, rain_reg_recent_sig, 'Regresi Rainfall vs Nino34 DJF P2', 'rainfall_reg_p2.png'),
#     (rain_reg_recent_minus_past, rain_reg_recent_minus_past_sig, 'Selisih Regresi Rainfall P2-P1', 'rainfall_reg_p2_minus_p1.png'),
# ]

# for data, sig_mask, title, filename in plot_defs:
#     fig = plt.figure(figsize=(10, 6))
#     ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

#     plot_data = smooth_for_plot(data).reset_coords(drop=True)
#     reg_levels, reg_ticks, _ = symmetric_levels(plot_data)
#     img = plot_data.plot.pcolormesh(
#         ax=ax,
#         transform=ccrs.PlateCarree(),
#         cmap='bwr',
#         levels=reg_levels,
#         extend='both',
#         add_colorbar=False,
#         add_labels=False,
#         infer_intervals=True,
#     )

#     sig_plot = (sig_mask.fillna(False).astype(np.int8).coarsen(lat=8, lon=8, boundary='trim').max() > 0)
#     yy, xx = np.where(sig_plot.values)
#     if yy.size > 0:
#         ax.scatter(
#             sig_plot['lon'].values[xx],
#             sig_plot['lat'].values[yy],
#             s=15,
#             c='k',
#             marker='.',
#             linewidths=0,
#             alpha=0.7,
#             transform=ccrs.PlateCarree(),
#             zorder=4,
#         )

#     ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
#     ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
#     ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
#     ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
#     ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
#     ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
#     ax.xaxis.set_major_formatter(LongitudeFormatter())
#     ax.yaxis.set_major_formatter(LatitudeFormatter())
#     ax.tick_params(direction='out', top=True, right=True, labelsize=14)
#     ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
#     ax.set_xlabel('')
#     ax.set_ylabel('')

#     cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
#     cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=reg_ticks, spacing='proportional', extend='both')
#     style_horizontal_cbar(cbar)
#     cbar.set_label('Slope regresi rainfall', fontsize=14, labelpad=10)
#     cbar.ax.tick_params(labelsize=14)

#     save_plot(fig, filename)
#     plt.show()
#     plt.close(fig)


# 4. Wind (vector) - U Shade


In [ ]:
wind_u_levels = split_period_levels(
    wind_u_reg_clim,
    wind_u_reg_past,
    wind_u_reg_recent,
    wind_u_reg_recent_minus_past,
    quantile=0.985,
    diff_quantile=0.98,
)
wind_vector_styles = split_vector_styles(
    [
        (wind_u_reg_clim, wind_v_reg_clim),
        (wind_u_reg_past, wind_v_reg_past),
        (wind_u_reg_recent, wind_v_reg_recent),
    ],
    [(wind_u_reg_recent_minus_past, wind_v_reg_recent_minus_past)],
    'm/s',
    quantile=0.90,
    key_fraction=0.50,
    target_ref_width=0.085,
)
plot_defs = [
    (wind_u_reg_clim, wind_u_reg_clim, wind_v_reg_clim, 'main', 'Regresi UWND vs Nino34 DJF 1981-2025', 'wind_u_reg_1981-2025.png'),
    (wind_u_reg_past, wind_u_reg_past, wind_v_reg_past, 'main', 'Regresi UWND vs Nino34 DJF P1', 'wind_u_reg_p1.png'),
    (wind_u_reg_recent, wind_u_reg_recent, wind_v_reg_recent, 'main', 'Regresi UWND vs Nino34 DJF P2', 'wind_u_reg_p2.png'),
    (wind_u_reg_recent_minus_past, wind_u_reg_recent_minus_past, wind_v_reg_recent_minus_past, 'diff', 'Selisih Regresi UWND P2-P1', 'wind_u_reg_p2_minus_p1.png'),
]
plot_vector_period_maps(
    plot_defs,
    wind_u_levels,
    wind_vector_styles,
    cbar_label='Slope regresi wind u',
    smooth_sigma=0.7,
    quiver_step=wind_step,
)


# 5. Wind (vector) - V Shade


In [ ]:
wind_v_levels = split_period_levels(
    wind_v_reg_clim,
    wind_v_reg_past,
    wind_v_reg_recent,
    wind_v_reg_recent_minus_past,
    quantile=0.985,
    diff_quantile=0.98,
)
wind_vector_styles = split_vector_styles(
    [
        (wind_u_reg_clim, wind_v_reg_clim),
        (wind_u_reg_past, wind_v_reg_past),
        (wind_u_reg_recent, wind_v_reg_recent),
    ],
    [(wind_u_reg_recent_minus_past, wind_v_reg_recent_minus_past)],
    'm/s',
    quantile=0.90,
    key_fraction=0.50,
    target_ref_width=0.085,
)
plot_defs = [
    (wind_v_reg_clim, wind_u_reg_clim, wind_v_reg_clim, 'main', 'Regresi VWND vs Nino34 DJF 1981-2025', 'wind_v_reg_1981-2025.png'),
    (wind_v_reg_past, wind_u_reg_past, wind_v_reg_past, 'main', 'Regresi VWND vs Nino34 DJF P1', 'wind_v_reg_p1.png'),
    (wind_v_reg_recent, wind_u_reg_recent, wind_v_reg_recent, 'main', 'Regresi VWND vs Nino34 DJF P2', 'wind_v_reg_p2.png'),
    (wind_v_reg_recent_minus_past, wind_u_reg_recent_minus_past, wind_v_reg_recent_minus_past, 'diff', 'Selisih Regresi VWND P2-P1', 'wind_v_reg_p2_minus_p1.png'),
]
plot_vector_period_maps(
    plot_defs,
    wind_v_levels,
    wind_vector_styles,
    cbar_label='Slope regresi wind v',
    smooth_sigma=0.7,
    quiver_step=wind_step,
)


# 6. MFC Regression


In [ ]:
mfc_plot_scale = 1e5
mfc_levels = {
    'main': {
        'levels': np.arange(-3, 3.0 + 0.25 / 2, 0.25),
        'ticks': np.arange(-3, 3.0 + 0.5 / 2, 0.5),
    },
    'diff': {
        'levels': np.arange(-3, 3.0 + 0.25 / 2, 0.25),
        'ticks': np.arange(-3, 3.0 + 0.5 / 2, 0.5),
    },
}
mfc_vector_styles = {
    'main': (1500, 50.0, '50 kg m$^{-2}$ s$^{-1}$'),
    'diff': (700, 30.0, '30 kg m$^{-2}$ s$^{-1}$'),
}
plot_defs = [
    (mfc_reg_clim * mfc_plot_scale, mfc_qx_reg_clim, mfc_qy_reg_clim, 'main', 'Regresi MFC vs Nino34 DJF 1981-2025', 'mfc_reg_1981-2025.png'),
    (mfc_reg_past * mfc_plot_scale, mfc_qx_reg_past, mfc_qy_reg_past, 'main', 'Regresi MFC vs Nino34 DJF P1', 'mfc_reg_p1.png'),
    (mfc_reg_recent * mfc_plot_scale, mfc_qx_reg_recent, mfc_qy_reg_recent, 'main', 'Regresi MFC vs Nino34 DJF P2', 'mfc_reg_p2.png'),
    (mfc_reg_recent_minus_past * mfc_plot_scale, mfc_qx_reg_recent_minus_past, mfc_qy_reg_recent_minus_past, 'diff', 'Selisih Regresi MFC P2-P1', 'mfc_reg_p2_minus_p1.png'),
]
plot_vector_period_maps(
    plot_defs,
    mfc_levels,
    mfc_vector_styles,
    cbar_label='Slope regresi MFC (x$10^{-5}$)',
    smooth_sigma=0.8,
    quiver_step=mfc_step,
)


# 7. Streamfunction Regression


In [ ]:
psi_plot_scale = 1e6
psi_levels = {
    'main': {
        'levels': np.arange(-2, 2.0 + 0.1 / 2, 0.1),
        'ticks': np.arange(-2, 2.0 + 0.5 / 2, 0.5),
    },
    'diff': {
        'levels': np.arange(-1, 1.0 + 0.05 / 2, 0.05),
        'ticks': np.arange(-1, 1.0 + 0.2 / 2, 0.2),
    },
}
psi_vector_styles = {
    'main': (30, 1, '1 m/s'),
    'diff': (15, 1, '1 m/s'),
}
plot_defs = [
    (psi_reg_clim / psi_plot_scale, u_psi_reg_clim, v_psi_reg_clim, 'main', 'Regresi Streamfunction vs Nino34 DJF 1981-2025', 'streamfunction_reg_1981-2025.png'),
    (psi_reg_past / psi_plot_scale, u_psi_reg_past, v_psi_reg_past, 'main', 'Regresi Streamfunction vs Nino34 DJF P1', 'streamfunction_reg_p1.png'),
    (psi_reg_recent / psi_plot_scale, u_psi_reg_recent, v_psi_reg_recent, 'main', 'Regresi Streamfunction vs Nino34 DJF P2', 'streamfunction_reg_p2.png'),
    (psi_reg_recent_minus_past / psi_plot_scale, u_psi_reg_recent_minus_past, v_psi_reg_recent_minus_past, 'diff', 'Selisih Regresi Streamfunction P2-P1', 'streamfunction_reg_p2_minus_p1.png'),
]
plot_vector_period_maps(
    plot_defs,
    psi_levels,
    psi_vector_styles,
    cbar_label='Slope regresi streamfunction (x$10^{6}$)',
    smooth_sigma=0.6,
    quiver_step=psi_chi_step,
)


# 8. Velocity Potential Regression


In [ ]:
chi_plot_scale = 1e6
chi_levels = {
    'main': {
        'levels': np.arange(-2, 2.0 + 0.1 / 2, 0.1),
        'ticks': np.arange(-2, 2.0 + 0.5 / 2, 0.5),
    },
    'diff': {
        'levels': np.arange(-1, 1.0 + 0.05 / 2, 0.05),
        'ticks': np.arange(-1, 1.0 + 0.2 / 2, 0.2),
    },
}
chi_vector_styles = {
    'main': (15, 0.5, '0.5 m/s'),
    'diff': (5, 0.5, '0.5 m/s'),
}
plot_defs = [
    (chi_reg_clim / chi_plot_scale, u_chi_reg_clim, v_chi_reg_clim, 'main', 'Regresi Velocity Potential vs Nino34 DJF 1981-2025', 'velocity_potential_reg_1981-2025.png'),
    (chi_reg_past / chi_plot_scale, u_chi_reg_past, v_chi_reg_past, 'main', 'Regresi Velocity Potential vs Nino34 DJF P1', 'velocity_potential_reg_p1.png'),
    (chi_reg_recent / chi_plot_scale, u_chi_reg_recent, v_chi_reg_recent, 'main', 'Regresi Velocity Potential vs Nino34 DJF P2', 'velocity_potential_reg_p2.png'),
    (chi_reg_recent_minus_past / chi_plot_scale, u_chi_reg_recent_minus_past, v_chi_reg_recent_minus_past, 'diff', 'Selisih Regresi Velocity Potential P2-P1', 'velocity_potential_reg_p2_minus_p1.png'),
]
plot_vector_period_maps(
    plot_defs,
    chi_levels,
    chi_vector_styles,
    cbar_label='Slope regresi velocity potential (x10^-6)',
    smooth_sigma=0.6,
    quiver_step=psi_chi_step,
)


# 9. MSLP Regression

In [ ]:

mslp_levels = {
    'main': {
        'levels': np.arange(-2.5, 2.5 + 0.1 / 2, 0.1),
        'ticks': np.arange(-2.5, 2.5 + 0.5 / 2, 0.5),
    },
    'diff': {
        'levels': np.arange(-1, 1 + 0.05 / 2, 0.05),
        'ticks': np.arange(-1, 1 + 0.2 / 2, 0.2),
    },
}
plot_defs = [
    (mslp_reg_clim / 100, mslp_reg_clim_sig, 'main', 'Regresi MSLP vs Nino34 DJF 1981-2025', 'mslp_reg_1981-2025.png'),
    (mslp_reg_past / 100, mslp_reg_past_sig, 'main', 'Regresi MSLP vs Nino34 DJF P1', 'mslp_reg_p1.png'),
    (mslp_reg_recent / 100, mslp_reg_recent_sig, 'main', 'Regresi MSLP vs Nino34 DJF P2', 'mslp_reg_p2.png'),
    (mslp_reg_recent_minus_past / 100, mslp_reg_recent_minus_past_sig, 'diff', 'Selisih Regresi MSLP P2-P1', 'mslp_reg_p2_minus_p1.png'),
]
plot_scalar_period_maps(
    plot_defs,
    mslp_levels,
    cbar_label='Slope regresi MSLP (hPa)',
    smooth_sigma=0.5,
    sig_step=8,
)


# 10. Geopotential Height 850 hPa Regression

In [ ]:
gh850_levels = {
    'main': {
        'levels': np.arange(-10, 10.0 + 0.5 / 2, 0.5),
        'ticks': np.arange(-10, 10.0 + 2 / 2, 2),
    },
    'diff': {
        'levels': np.arange(-2, 2.0 + 0.1 / 2, 0.1),
        'ticks': np.arange(-2, 2.0 + 0.5 / 2, 0.5),
    },
}
plot_defs = [
    (gh850_reg_clim, gh850_reg_clim_sig, 'main', 'Regresi Geopotential Height 850 hPa vs Nino34 DJF 1981-2025', 'geopotential_height_850_reg_1981-2025.png'),
    (gh850_reg_past, gh850_reg_past_sig, 'main', 'Regresi Geopotential Height 850 hPa vs Nino34 DJF P1', 'geopotential_height_850_reg_p1.png'),
    (gh850_reg_recent, gh850_reg_recent_sig, 'main', 'Regresi Geopotential Height 850 hPa vs Nino34 DJF P2', 'geopotential_height_850_reg_p2.png'),
    (gh850_reg_recent_minus_past, gh850_reg_recent_minus_past_sig, 'diff', 'Selisih Regresi Geopotential Height 850 hPa P2-P1', 'geopotential_height_850_reg_p2_minus_p1.png'),
]
plot_scalar_period_maps(
    plot_defs,
    gh850_levels,
    cbar_label='Slope regresi geopotential height 850 hPa',
    smooth_sigma=0.5,
    sig_step=8,
)


# 11. Streamfunction Shading with Wind Vectors


In [ ]:
plot_defs = [
    (psi_reg_clim / psi_plot_scale, wind_u_reg_clim, wind_v_reg_clim, 'main', 'Regresi Streamfunction vs Nino34 DJF 1981-2025', 'streamfunction_wind_reg_1981-2025.png'),
    (psi_reg_past / psi_plot_scale, wind_u_reg_past, wind_v_reg_past, 'main', 'Regresi Streamfunction vs Nino34 DJF P1', 'streamfunction_wind_reg_p1.png'),
    (psi_reg_recent / psi_plot_scale, wind_u_reg_recent, wind_v_reg_recent, 'main', 'Regresi Streamfunction vs Nino34 DJF P2', 'streamfunction_wind_reg_p2.png'),
    (psi_reg_recent_minus_past / psi_plot_scale, wind_u_reg_recent_minus_past, wind_v_reg_recent_minus_past, 'diff', 'Selisih Regresi Streamfunction P2-P1', 'streamfunction_wind_reg_p2_minus_p1.png'),
]
plot_vector_period_maps(
    plot_defs,
    psi_levels,
    wind_vector_styles,
    cbar_label='Slope regresi streamfunction (x$10^{6}$)',
    smooth_sigma=0.6,
    quiver_step=wind_step,
)


# 12. Velocity Potential Shading with Wind Vectors


In [ ]:
plot_defs = [
    (chi_reg_clim / chi_plot_scale, wind_u_reg_clim, wind_v_reg_clim, 'main', 'Regresi Velocity Potential vs Nino34 DJF 1981-2025', 'velocity_potential_wind_reg_1981-2025.png'),
    (chi_reg_past / chi_plot_scale, wind_u_reg_past, wind_v_reg_past, 'main', 'Regresi Velocity Potential vs Nino34 DJF P1', 'velocity_potential_wind_reg_p1.png'),
    (chi_reg_recent / chi_plot_scale, wind_u_reg_recent, wind_v_reg_recent, 'main', 'Regresi Velocity Potential vs Nino34 DJF P2', 'velocity_potential_wind_reg_p2.png'),
    (chi_reg_recent_minus_past / chi_plot_scale, wind_u_reg_recent_minus_past, wind_v_reg_recent_minus_past, 'diff', 'Selisih Regresi Velocity Potential P2-P1', 'velocity_potential_wind_reg_p2_minus_p1.png'),
]
plot_vector_period_maps(
    plot_defs,
    chi_levels,
    wind_vector_styles,
    cbar_label='Slope regresi velocity potential (x10^-6)',
    smooth_sigma=0.6,
    quiver_step=wind_step,
)


# 13. MSLP Shading with Wind Vectors


In [ ]:
plot_defs = [
    (mslp_reg_clim / 100, wind_u_reg_clim, wind_v_reg_clim, 'main', 'Regresi MSLP vs Nino34 DJF 1981-2025', 'mslp_wind_reg_1981-2025.png'),
    (mslp_reg_past / 100, wind_u_reg_past, wind_v_reg_past, 'main', 'Regresi MSLP vs Nino34 DJF P1', 'mslp_wind_reg_p1.png'),
    (mslp_reg_recent / 100, wind_u_reg_recent, wind_v_reg_recent, 'main', 'Regresi MSLP vs Nino34 DJF P2', 'mslp_wind_reg_p2.png'),
    (mslp_reg_recent_minus_past / 100, wind_u_reg_recent_minus_past, wind_v_reg_recent_minus_past, 'diff', 'Selisih Regresi MSLP P2-P1', 'mslp_wind_reg_p2_minus_p1.png'),
]
plot_vector_period_maps(
    plot_defs,
    mslp_levels,
    wind_vector_styles,
    cbar_label='Slope regresi MSLP (hPa)',
    smooth_sigma=0.5,
    quiver_step=wind_step,
)
